# MicroGrad — End-to-End Demonstration

**Course:** CSE485–CSE488 Deep Learning  
**Assignment 1:** MicroGrad (Scalar Autograd Engine)

This notebook demonstrates a complete experiment using the assignment implementation:
1. Generate and save a small dataset, then load it from CSV.
2. Build an MLP using the project's `Neuron`, `Layer`, and `MLP` classes.
3. Train the model using gradients computed by MicroGrad and manual gradient descent.
4. Evaluate the model and visualize the training loss and decision boundary.

> Run this notebook from the assignment repository root (the folder containing `src/`). The notebook imports your implementation rather than reimplementing the engine here.

## 1. Environment and imports

The model code is imported from `src/nn.py`. NumPy and Matplotlib are used for data handling and visualization; model operations and gradients are computed with MicroGrad.

In [ ]:
import csv
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Support running from either the repository root or notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.nn import MLP

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("MicroGrad MLP imported successfully.")

## 2. Create and load a dataset

We use a deterministic two-class concentric-circles dataset. Targets are `-1` and `+1`, suitable for a `tanh` output neuron.

The CSV is generated locally and then loaded back from disk, making the data-loading step reproducible without an external download.

In [ ]:
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "micrograd_circles.csv"

rng = np.random.default_rng(SEED)
n_per_class = 100

inner_angles = rng.uniform(0, 2 * np.pi, n_per_class)
inner_radii = rng.normal(loc=0.35, scale=0.035, size=n_per_class)
inner_x = np.column_stack([inner_radii * np.cos(inner_angles),
                           inner_radii * np.sin(inner_angles)])
inner_y = np.full(n_per_class, -1.0)

outer_angles = rng.uniform(0, 2 * np.pi, n_per_class)
outer_radii = rng.normal(loc=0.85, scale=0.045, size=n_per_class)
outer_x = np.column_stack([outer_radii * np.cos(outer_angles),
                           outer_radii * np.sin(outer_angles)])
outer_y = np.full(n_per_class, 1.0)

X_all = np.vstack([inner_x, outer_x])
y_all = np.concatenate([inner_y, outer_y])
order = rng.permutation(len(y_all))
X_all, y_all = X_all[order], y_all[order]

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["x1", "x2", "label"])
    writer.writerows([[float(a), float(b), float(label)]
                      for (a, b), label in zip(X_all, y_all)])

print(f"Saved {len(y_all)} examples to {csv_path}")

In [ ]:
loaded_rows = []
with csv_path.open("r", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        loaded_rows.append((float(row["x1"]), float(row["x2"]), float(row["label"])))

X = np.array([[r[0], r[1]] for r in loaded_rows], dtype=float)
y = np.array([r[2] for r in loaded_rows], dtype=float)

split_rng = np.random.default_rng(SEED)
indices = split_rng.permutation(len(y))
train_size = int(0.8 * len(y))
train_idx, test_idx = indices[:train_size], indices[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Loaded dataset:", X.shape)
print("Training examples:", len(train_idx))
print("Testing examples:", len(test_idx))
print("Labels:", np.unique(y))

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm",
            edgecolors="black", s=28)
plt.title("Training data: concentric circles")
plt.xlabel("x1")
plt.ylabel("x2")
plt.axis("equal")
plt.grid(alpha=0.25)
plt.show()

## 3. Build the neural network

Architecture:
- Input: 2 features
- Hidden layer 1: 16 neurons
- Hidden layer 2: 16 neurons
- Output: 1 neuron

This uses the standard MicroGrad MLP constructor: `MLP(nin, nouts)`, where `nouts` is the list of layer widths.

In [ ]:
model = MLP(2, [16, 16, 1])

def as_scalar(output):
    # Standard MicroGrad MLPs may return a Value or a list of Values.
    if isinstance(output, (list, tuple)):
        if len(output) != 1:
            raise ValueError(f"Expected one output, received {len(output)}")
        output = output[0]
    return output

sample_output = as_scalar(model([float(X_train[0, 0]), float(X_train[0, 1])]))
print("Output type:", type(sample_output).__name__)
print("Initial prediction:", sample_output.data)
print("Number of trainable parameters:", len(model.parameters()))

## 4. Define loss and evaluation helpers

We use mean squared error (MSE) between the network's output and targets (`-1` or `+1`).

For each training iteration:
1. Reset parameter gradients.
2. Run the forward pass and construct the loss graph.
3. Call `backward()` on the loss.
4. Update parameters using gradient descent.

The derivatives are computed by MicroGrad, not NumPy.

In [ ]:
def predict_one(features):
    return as_scalar(model([float(features[0]), float(features[1])]))

def mse_loss(features, targets):
    total = None
    for features_i, target_i in zip(features, targets):
        error = predict_one(features_i) - float(target_i)
        example_loss = error ** 2
        total = example_loss if total is None else total + example_loss
    return total / len(features)

def predict_array(features):
    return np.array([predict_one(row).data for row in features], dtype=float)

def accuracy(features, targets):
    predictions = predict_array(features)
    predicted_labels = np.where(predictions >= 0.0, 1.0, -1.0)
    return float(np.mean(predicted_labels == targets))

## 5. Train the MLP

This uses full-batch gradient descent. The number of epochs is intentionally modest because MicroGrad builds an explicit scalar computation graph.

In [ ]:
learning_rate = 0.03
epochs = 60
history = []
parameters = model.parameters()

for epoch in range(epochs):
    for parameter in parameters:
        parameter.grad = 0.0

    loss = mse_loss(X_train, y_train)
    loss.backward()

    for parameter in parameters:
        parameter.data -= learning_rate * parameter.grad

    history.append(float(loss.data))
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:3d}/{epochs} | loss = {loss.data:.6f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), history)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.grid(alpha=0.3)
plt.show()

## 6. Evaluate the trained model

We report classification accuracy on the held-out test set. The sign of the scalar output determines the predicted class.

In [ ]:
test_predictions = predict_array(X_test)
test_labels = np.where(test_predictions >= 0.0, 1.0, -1.0)
test_accuracy = float(np.mean(test_labels == y_test))
test_mse = float(np.mean((test_predictions - y_test) ** 2))

print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test MSE:      {test_mse:.6f}")

## 7. Visualize the decision boundary

The plot evaluates the trained MLP over a grid of 2D points. The background shows the model's continuous output, while the points show held-out test examples.

In [ ]:
padding = 0.15
x1_min, x1_max = X[:, 0].min() - padding, X[:, 0].max() + padding
x2_min, x2_max = X[:, 1].min() - padding, X[:, 1].max() + padding

grid_x1, grid_x2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 90),
    np.linspace(x2_min, x2_max, 90)
)
grid_points = np.column_stack([grid_x1.ravel(), grid_x2.ravel()])

grid_outputs = []
chunk_size = 100
for start in range(0, len(grid_points), chunk_size):
    grid_outputs.extend(predict_array(grid_points[start:start + chunk_size]))

Z = np.array(grid_outputs).reshape(grid_x1.shape)

plt.figure(figsize=(7, 6))
plt.contourf(grid_x1, grid_x2, Z, levels=20, cmap="coolwarm", alpha=0.65)
plt.contour(grid_x1, grid_x2, Z, levels=[0.0], colors="black", linewidths=2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm",
            edgecolors="black", s=38)
plt.title(f"MicroGrad MLP decision boundary (test accuracy: {test_accuracy:.1%})")
plt.xlabel("x1")
plt.ylabel("x2")
plt.axis("equal")
plt.grid(alpha=0.2)
plt.show()

## 8. Conclusion

This experiment demonstrates the complete MicroGrad workflow:
- Scalar values and operations build a computation graph.
- `backward()` propagates gradients through that graph.
- The MLP API composes neurons and layers into a trainable network.
- Gradient descent updates model parameters.
- The notebook loads data, trains the model, evaluates it, and visualizes results.

### Reproducibility notes
- The random seed is fixed (`42`).
- The dataset is generated locally and saved to `data/micrograd_circles.csv`.
- The notebook imports `MLP` from the assignment's `src/nn.py`.
- Run all cells from top to bottom after activating the project environment.

### Notebook dependencies

Install these if they are not already available:

```bash
python -m pip install numpy matplotlib ipykernel
```

These notebook dependencies are separate from the GitHub autograder dependencies.